# Day 016 — Exercise 5: The Streaming Chatbot

**Goal:** Implement `run_streaming_chatbot(system_prompt, model, max_turns)` — the streaming version of `run_chatbot` from Day 15. Use `stream_chat_turn` instead of `chat_turn` and print `'Bot: '` before each streaming reply.

In [ ]:
import ollama
import io
import sys

In [ ]:
def stream_tokens(messages: list[dict], model: str = "llama3.2"):
    """Yield tokens one at a time from a streaming Ollama response."""
    response = ollama.chat(model=model, messages=messages, stream=True)
    for chunk in response:
        yield chunk["message"]["content"]

In [ ]:
def collect_stream(tokens) -> str:
    """Consume a token iterator and return the full reply as a string."""
    return "".join(tokens)

In [ ]:
def print_stream(tokens) -> None:
    """Print tokens to stdout as they arrive; final newline at end."""
    for token in tokens:
        print(token, end="", flush=True)
    print()

In [ ]:
def append_turn(history: list[dict], user_text: str, assistant_text: str) -> list[dict]:
    """Return a new history list with one user+assistant turn appended."""
    return history + [
        {"role": "user", "content": user_text},
        {"role": "assistant", "content": assistant_text},
    ]

In [ ]:
def stream_chat_turn(
    history: list[dict],
    user_input: str,
    model: str = "llama3.2",
) -> tuple[str, list[dict]]:
    """Stream the model reply to stdout and return (reply, new_history)."""
    messages = history + [{"role": "user", "content": user_input}]
    tokens = stream_tokens(messages, model)
    parts = []
    for token in tokens:
        print(token, end="", flush=True)
        parts.append(token)
    print()
    reply = "".join(parts)
    return reply, append_turn(history, user_input, reply)

In [ ]:
def truncate_history(history: list[dict], max_turns: int = 10) -> list[dict]:
    """Keep the system prompt and the last max_turns*2 non-system messages."""
    if not history:
        return []
    if history[0]["role"] == "system":
        system, tail = [history[0]], history[1:]
    else:
        system, tail = [], history
    return system + tail[-(max_turns * 2):]

In [ ]:
def reset_history(history: list[dict]) -> list[dict]:
    """Return a new history containing only the system prompt (if present)."""
    if history and history[0]["role"] == "system":
        return [history[0]]
    return []

In [ ]:
def format_history(history: list[dict]) -> str:
    """Render conversation history as a readable transcript."""
    labels = {"user": "You", "assistant": "Bot"}
    lines = []
    for msg in history:
        if msg["role"] == "system":
            continue
        label = labels.get(msg["role"], msg["role"].capitalize())
        lines.append(f"{label}: {msg['content']}")
    return "\n".join(lines)

## Your Implementation

In [ ]:
SYSTEM_PROMPT = "You are a helpful, concise assistant. Answer clearly and briefly."

def run_streaming_chatbot(
    system_prompt: str = SYSTEM_PROMPT,
    model: str = "llama3.2",
    max_turns: int = 10,
) -> None:
    """
    Run a streaming multi-turn CLI chatbot until the user types /quit.

    Identical to run_chatbot from Day 15 except:
      - print('Bot: ', end='', flush=True) before each turn
      - stream_chat_turn replaces chat_turn
      - no separate print for the reply (stream_chat_turn handles printing)
    """
    # TODO: init history with system prompt
    # TODO: print welcome banner
    # TODO: while True: read input, handle /quit /reset /history, stream reply
    pass

## Check Your Work

In [ ]:
import io, sys

def _run_checks():
    total = 5
    passed = 0

    # Check 1: all five streaming functions are defined
    try:
        for name in ["stream_tokens", "collect_stream", "print_stream",
                     "stream_chat_turn", "run_streaming_chatbot"]:
            assert name in globals(), f"{name} not defined"
        passed += 1; print("✅ Check 1: all five streaming functions defined")
    except Exception as e:
        print(f"❌ Check 1: missing function — {e}")

    # Check 2: stream_chat_turn produces a non-empty reply
    try:
        history = [{"role": "system", "content": "You are a helpful assistant."}]
        old = sys.stdout; sys.stdout = io.StringIO()
        reply, history = stream_chat_turn(history, "Say the number: 42")
        sys.stdout = old
        assert isinstance(reply, str) and len(reply.strip()) > 0
        passed += 1; print("✅ Check 2: stream_chat_turn produces a non-empty reply")
    except Exception as e:
        sys.stdout = old
        print(f"❌ Check 2: stream_chat_turn — {e}")

    # Check 3: history has correct length after one turn
    try:
        assert len(history) == 3, \
            f"expected 3 messages (system+user+assistant), got {len(history)}"
        passed += 1; print("✅ Check 3: history has correct length after one turn")
    except Exception as e:
        print(f"❌ Check 3: history length — {e}")

    # Check 4: format_history works on streaming history
    try:
        transcript = format_history(history)
        assert isinstance(transcript, str)
        assert "You:" in transcript and "Bot:" in transcript
        passed += 1; print("✅ Check 4: format_history works on streaming history")
    except Exception as e:
        print(f"❌ Check 4: format_history — {e}")

    # Check 5: reset_history preserves only system prompt
    try:
        reset = reset_history(history)
        assert len(reset) == 1 and reset[0]["role"] == "system"
        passed += 1; print("✅ Check 5: reset_history preserves only system prompt")
    except Exception as e:
        print(f"❌ Check 5: reset_history — {e}")

    if passed == total:
        print("🎉 Exercise complete!")
    print(f"\nScore: {passed}/{total}")

_run_checks()

## Solution

<details>
<summary>Click to reveal</summary>

```python
SYSTEM_PROMPT = "You are a helpful, concise assistant. Answer clearly and briefly."

def run_streaming_chatbot(
    system_prompt: str = SYSTEM_PROMPT,
    model: str = "llama3.2",
    max_turns: int = 10,
) -> None:
    history = [{"role": "system", "content": system_prompt}]
    print("Streaming chatbot ready. Commands: /quit  /reset  /history")
    print("-" * 50)

    while True:
        try:
            user_input = input("You: ").strip()
        except (EOFError, KeyboardInterrupt):
            print("\nGoodbye!")
            break

        if not user_input:
            continue

        if user_input.startswith("/"):
            if user_input == "/quit":
                print("Goodbye!")
                break
            elif user_input == "/reset":
                history = reset_history(history)
                print("Bot: Conversation reset.")
            elif user_input == "/history":
                transcript = format_history(history)
                print(transcript if transcript else "(no history yet)")
            else:
                print(f"Bot: Unknown command: {user_input}")
            continue

        print("Bot: ", end="", flush=True)
        reply, history = stream_chat_turn(history, user_input, model)
        history = truncate_history(history, max_turns)
```

</details>